# Creare una Libreria Attuariale di Tariffazione Riutilizzabile con PROC FCMP

## Sintesi Esecutiva

Un assicuratore property-and-casualty incapsula la propria matematica di tariffazione — premio puro, caricamento per spese/rischio, ponderazione di credibilità a fluttuazione limitata e stima della riserva attualizzata — come funzioni personalizzate e una subroutine multi-output in **PROC FCMP**. La libreria compilata viene registrata tramite l'opzione di sistema `CMPLIB=` e poi richiamata riga per riga da uno step DATA che tariffa un portafoglio sintetico di 100 polizze. Ogni valore tariffato in questo notebook — il fattore di credibilità `z`, il premio puro combinato, il premio caricato e la riserva sinistri attualizzata — è prodotto da queste routine compilate, non da un'aritmetica inline. Il risultato: il rapporto sinistri implicito si attesta al 60.5% (regione rurale), 55.8% (regione suburbana) e 63.6% (regione urbana) — comodamente sotto il 100% in ogni segmento, a conferma che il premio caricato copre la perdita attesa mentre lo step di tariffazione resta pulito e verificabile.

## Fonti dei Dati

| Dataset | Righe | Descrizione | Variabili Chiave |
|---------|------|-------------|---------------|
| `work.portfolio` | 100 | Portafoglio sintetico di polizze P&C in vigore, generato inline con `rand()` | `policy_id`, `region` (urbana/suburbana/rurale), `years_insured`, `exposure` (anni-auto), `claim_count` (Poisson), `incurred_loss` (severità gamma x conteggio), `class_pure_prem` (tariffa manuale per regione) |

Lo step DATA itera su un intervallo di `policy_id` più ampio, ma questo ambiente funziona senza licenza, quindi l'output è limitato alle prime **100 osservazioni** — il libro tariffato qui sotto riflette proprio queste 100 polizze.

# Creare una Libreria Attuariale di Tariffazione Riutilizzabile con PROC FCMP

Gli attuari ripetono gli stessi calcoli in lavori di tariffazione, riservazione e reporting: convertire le perdite in un *premio puro*, applicare *caricamenti per spese e rischio* per arrivare al premio caricato, combinare l'esperienza propria di una singola polizza con una tariffa di classe usando la *credibilità*, e *attualizzare* i flussi di cassa futuri a valore presente. Riscrivere queste formule in ogni step DATA espone a errori di copia-incolla e rende dolorosi i cambi di assunzioni.

**PROC FCMP** (il compilatore di funzioni SAS) ci permette di definire ogni formula una sola volta come funzione o subroutine con nome, memorizzare le routine compilate in una libreria, e poi richiamarle come qualsiasi funzione SAS predefinita. In questo notebook:

1. Compiliamo una piccola libreria di funzioni attuariali con `PROC FCMP`.
2. La registriamo per la sessione con l'opzione di sistema `CMPLIB=`.
3. Generiamo un portafoglio sintetico property-and-casualty.
4. Tariffiamo ogni polizza richiamando le nostre funzioni personalizzate e una subroutine multi-output da un singolo step DATA.
5. Riepiloghiamo e interpretiamo il portafoglio tariffato.

## Passo 1 — Generare un Portafoglio Sintetico di Polizze

Simuliamo un libro di polizze auto in vigore (le prime 100 vengono tariffate qui sotto per il limite della modalità senza licenza). Ogni polizza appartiene a una regione tariffaria con il proprio *premio puro* manuale (perdita attesa per anno-auto). Il numero di sinistri segue un processo di Poisson scalato per l'esposizione, e le severità seguono una distribuzione gamma, quindi `incurred_loss` è una perdita composta realistica (Poisson x gamma). `years_insured` guiderà in seguito il peso di credibilità. Un seed fisso tramite `call streaminit` rende i dati riproducibili.

In [1]:
DATI portfolio;
    CHIAMARE streaminit(20260531);
    LUNGHEZZA region $10;
    FARE policy_id = 10001 FINO_A 12000;
        /* Assign a rating region */
        u = rand('uniform');
        SE_COND u < 0.40 ALLORA FARE; region = 'Urbana';    base_pp = 820; lambda = 0.18; FINE;
        ALTRIMENTI SE_COND u < 0.75 ALLORA FARE; region = 'Suburbana'; base_pp = 540; lambda = 0.11; FINE;
        ALTRIMENTI FARE; region = 'Rurale';    base_pp = 360; lambda = 0.07; FINE;

        /* Tenure (years insured) and exposure (car-years) */
        years_insured = 1 + rand('poisson', 3);
        exposure = round(0.5 + rand('gamma', 4) / 4, 0.01);

        /* Compound claim process: Poisson frequency, gamma severity */
        claim_count = rand('poisson', lambda * exposure);
        incurred_loss = 0;
        FARE c = 1 FINO_A claim_count;
            incurred_loss = incurred_loss + rand('gamma', 2.0) * 2500;
        FINE;
        incurred_loss = round(incurred_loss, 0.01);

        /* Manual class pure premium for this policy's exposure */
        class_pure_prem = round(base_pp * exposure, 0.01);

        USCITA;
    FINE;
    MANTENERE policy_id region years_insured exposure claim_count
         incurred_loss class_pure_prem;
ESEGUIRE;

PROCEDURA STAMPARE DATI=portfolio(obs=8) noobs ETICHETTA;
    ETICHETTA policy_id="ID Polizza" region="Regione" years_insured="Anni Assicurato"
          exposure="Esposizione (anni-auto)" claim_count="Numero Sinistri"
          incurred_loss="Sinistri Incorsi" class_pure_prem="Premio Puro di Classe";
    TITOLO "Prime 8 Polizze Simulate";
ESEGUIRE;

                                                Prime 8 Polizze Simulate                                                

ID Polizza    Regione  Anni Assicurato  Esposizione (anni-auto)  Numero Sinistri  Sinistri Incorsi  Premio Puro di Classe
     10001  Rurale                   5                     1.36                0                 0                  489.6
     10002  Urbana                   8                     0.97                1           3432.28                  795.4
     10003  Urbana                   2                     1.53                2           7155.92                 1254.6
     10004  Suburbana                9                      2.4                0                 0                   1296
     10005  Rurale                   5                     0.99                0                 0                  356.4
     10006  Suburbana                1                     0.83                0                 0                  448.2
     10007  Rurale      


NOTE: DATA portfolio

NOTE: Unlicensed mode - output limited to 100 observations.

NOTE: Wrote portfolio (100 rows, 7 columns).
NOTE: DATA elapsed:
  wall  0.42 seconds
  cpu   0.42 seconds
NOTE: PROC PRINT data=portfolio

NOTE: PROC PRINT completed: 8 observations printed, 7 variables


## Passo 2 — Compilare la Libreria di Funzioni Attuariali

Ora il cuore del notebook. `PROC FCMP` con `OUTLIB=work.actfuncs.pricing` compila quattro funzioni e una subroutine nel pacchetto `pricing` del dataset `work.actfuncs`:

- **`pure_premium`** — perdita osservata per unità di esposizione (frequenza x severità combinate).
- **`credibility_z`** — fattore di credibilità a fluttuazione limitata `Z = sqrt(n / (n + k))`, dove `n` sono gli anni di esposizione della polizza e `k` è una costante di calibrazione.
- **`charged_premium`** — applica un caricamento di rischio proporzionale e un rapporto spese fisso a un costo sinistri per arrivare al premio effettivamente addebitato.
- **`pv_reserve`** — valore presente di un futuro pagamento sinistri, `FV / (1+r)**t`, usato per attualizzare le riserve sinistri.
- **`blend_premium`** (una SUBROUTINE) — usa `OUTARGS` per restituire *due* valori insieme: il premio puro ponderato per credibilità e il fattore di credibilità usato, così lo step DATA li ottiene entrambi con una sola chiamata.

Ogni routine si chiude con `ENDSUB`, e la subroutine indica i suoi argomenti scrivibili con `OUTARGS`.

In [2]:
PROCEDURA FCMP outlib=work.actfuncs.pricing;

    /* Observed pure premium: loss cost per unit of exposure */
    function pure_premium(loss, exposure);
        SE_COND exposure <= 0 ALLORA RETURN(.);
        RETURN(loss / exposure);
    endsub;

    /* Limited-fluctuation credibility: Z = sqrt(n / (n + k)) */
    function credibility_z(n_years, k);
        SE_COND n_years <= 0 ALLORA RETURN(0);
        RETURN(sqrt(n_years / (n_years + k)));
    endsub;

    /* Charged premium = loss cost grossed up for risk load + expenses */
    function charged_premium(loss_cost, risk_load, expense_ratio);
        gross = loss_cost * (1 + risk_load);
        SE_COND expense_ratio >= 1 ALLORA RETURN(.);
        RETURN(gross / (1 - expense_ratio));
    endsub;

    /* Present value of a future claim payment discounted t years at rate r */
    function pv_reserve(future_value, r, t);
        RETURN(future_value / (1 + r) ** t);
    endsub;

    /* Credibility blend: returns the weighted pure premium AND the Z used */
    subroutine blend_premium(own_pp, class_pp, n_years, k, blended, z_used);
        outargs blended, z_used;
        z_used = credibility_z(n_years, k);
        blended = z_used * own_pp + (1 - z_used) * class_pp;
    endsub;

ESEGUIRE;


               The FCMP Procedure

  Output Library: work.actfuncs.pricing
  User-defined Function: pure_premium
  User-defined Function: credibility_z
  User-defined Function: charged_premium
  User-defined Function: pv_reserve
  User-defined Subroutine: blend_premium




NOTE: PROC FCMP outlib=work.actfuncs.pricing

NOTE: Function pure_premium stored to work.actfuncs.pricing.
NOTE: Function credibility_z stored to work.actfuncs.pricing.
NOTE: Function charged_premium stored to work.actfuncs.pricing.
NOTE: Function pv_reserve stored to work.actfuncs.pricing.
NOTE: Subroutine blend_premium stored to work.actfuncs.pricing.


## Passo 3 — Registrare la Libreria con CMPLIB=

Compilare le routine non basta; occorre indicare a SAS dove trovarle quando un successivo step DATA o una procedura fa riferimento a un nome che non riconosce come predefinito. L'opzione di sistema `CMPLIB=` punta al dataset (non al nome del pacchetto a tre livelli) che contiene il codice compilato. Dopo questa istruzione `OPTIONS`, ogni funzione e subroutine descritta sopra è richiamabile per nome.

In [3]:
OPZIONI cmplib=work.actfuncs;


NOTE: Option CMPLIB changed to WORK.ACTFUNCS.


## Passo 4 — Tariffare il Portafoglio con le Funzioni Personalizzate

Lo step DATA di tariffazione è ora quasi privo di aritmetica — l'intento attuariale si legge direttamente dai nomi delle funzioni. Per ogni polizza:

1. Calcoliamo il premio puro osservato proprio della polizza con `pure_premium`.
2. Richiamiamo la subroutine `blend_premium` per ponderarlo per credibilità rispetto alla tariffa di classe regionale, recuperando sia il costo sinistri combinato sia il fattore di credibilità `z` tramite `OUTARGS`.
3. Portiamo il costo sinistri combinato al premio caricato con un caricamento di rischio del 12% e un rapporto spese del 25% tramite `charged_premium`.
4. Stimiamo la riserva sinistri ancora aperta come il 35% dei sinistri incorsi della polizza e la attualizziamo di tre anni al 4% con `pv_reserve`.

Nota come gli argomenti di output della subroutine (`blended_pp`, `z`) debbano essere inizializzati prima della `CALL`. La riserva attualizzata varia da polizza a polizza perché è determinata dai sinistri incorsi effettivi di ciascuna — le polizze senza sinistri portano una riserva pari a zero, quindi anche il loro `reserve_pv` è zero.

In [4]:
DATI rated;
    IMPOSTARE portfolio;

    /* 1. Policy's own loss experience as a pure premium */
    own_pp = round(pure_premium(incurred_loss, exposure), 0.01);

    /* 2. Credibility-weight own experience against the class rate.
          k = 4 exposure-years for full-ish credibility. */
    blended_pp = .;
    z = .;
    CHIAMARE blend_premium(own_pp, class_pure_prem / exposure,
                       years_insured, 4, blended_pp, z);
    blended_pp = round(blended_pp, 0.01);
    z = round(z, 0.001);

    /* 3. Convert blended loss cost (per car-year) to charged premium */
    loss_cost = blended_pp * exposure;
    premium = round(charged_premium(loss_cost, 0.12, 0.25), 0.01);

    /* 4. Outstanding case reserve = 35% of incurred loss, expected to
          settle in 3 years; discount it to present value at 4%. */
    case_reserve = round(0.35 * incurred_loss, 0.01);
    reserve_pv   = round(pv_reserve(case_reserve, 0.04, 3), 0.01);

    MANTENERE policy_id region years_insured exposure incurred_loss
         own_pp class_pure_prem blended_pp z premium
         case_reserve reserve_pv;
ESEGUIRE;

PROCEDURA STAMPARE DATI=rated(obs=10) noobs ETICHETTA;
    ETICHETTA policy_id="ID Polizza" region="Regione" years_insured="Anni Assicurato"
          exposure="Esposizione (anni-auto)" own_pp="Premio Puro Proprio"
          blended_pp="Premio Puro Combinato" z="Fattore di Credibilità"
          premium="Premio" reserve_pv="Riserva Attualizzata";
    TITOLO "Portafoglio Tariffato (prime 10 polizze)";
    VARIABILE policy_id region years_insured exposure own_pp
        blended_pp z premium reserve_pv;
ESEGUIRE;

                                        Portafoglio Tariffato (prime 10 polizze)                                        

ID Polizza    Regione  Anni Assicurato  Esposizione (anni-auto)  Premio Puro Proprio  Premio Puro Combinato   Fattore di Credibilità   Premio  Riserva Attualizzata
     10001  Rurale                   5                     1.36                    0                  91.67                    0.745   186.18                     0
     10002  Urbana                   8                     0.97              3538.43                3039.59                    0.816  4402.95               1067.95
     10003  Urbana                   2                     1.53              4677.07                3046.88                    0.577  6961.51               2226.55
     10004  Suburbana                9                      2.4                    0                  90.69                    0.832   325.03                     0
     10005  Rurale                   5                    


NOTE: DATA rated


NOTE: Read 100 rows from portfolio.
NOTE: Wrote rated (100 rows, 12 columns).
NOTE: DATA elapsed:
  wall  0.06 seconds
  cpu   0.06 seconds
NOTE: PROC PRINT data=rated

NOTE: PROC PRINT completed: 10 observations printed, 9 variables


## Passo 5 — Riepilogare il Libro Tariffato

Con ogni polizza tariffata tramite la stessa libreria compilata, possiamo aggregare il portafoglio per regione. Riportiamo il premio caricato medio, il fattore di credibilità medio, i sinistri incorsi totali e la riserva sinistri attualizzata totale, per poi derivare il rapporto sinistri implicito per segmento e verificare se il premio caricato copre la perdita attesa.

In [5]:
PROCEDURA MEDIE DATI=rated mean sum maxdec=2 nonobs;
    CLASSE region;
    VARIABILE premium z incurred_loss reserve_pv;
    ETICHETTA region="Regione" premium="Premio" z="Fattore di Credibilità"
          incurred_loss="Sinistri Incorsi" reserve_pv="Riserva Attualizzata";
    TITOLO "Riepilogo del Portafoglio per Regione Tariffaria";
ESEGUIRE;

/* Implied loss ratio = incurred loss / charged premium, plus the
   present-valued reserve the segment is carrying, by region. */
PROCEDURA SQL;
    TITOLO "Rapporto Sinistri Implicito e Riserva Attualizzata per Regione";
    SELEZIONARE region,
           count(*)                              AS n_policies,
           sum(incurred_loss)                    AS total_loss     FORMATO=dollar12.2,
           sum(premium)                          AS total_premium  FORMATO=dollar12.2,
           sum(incurred_loss) / sum(premium)     AS loss_ratio     FORMATO=percent8.1,
           sum(reserve_pv)                       AS total_pv_reserve FORMATO=dollar12.2
    FROM rated
    GROUP PER region
    ORDER PER region;
QUIT;
TITOLO;

                                    Riepilogo del Portafoglio per Regione Tariffaria                                    

                                                  The MEANS Procedure

                                           Analysis Variable : premium Premio

        Regione             Mean            Sum
        ---------------------------------------
        Rurale            476.61       11438.62
        Suburbana         813.04       34147.74
        Urbana           1987.17       67563.93
        ---------------------------------------

                                     Analysis Variable : z Fattore di Credibilità

        Regione             Mean            Sum
        ---------------------------------------
        Rurale              0.71          17.14
        Suburbana           0.68          28.36
        Urbana              0.70          23.90
        ---------------------------------------

                                   Analysis Variable : incurred_los


NOTE: PROC MEANS
NOTE: PROC MEANS statement used.
NOTE: PROC SQL 

NOTE: PROC SQL statement used.


## Interpretare i Risultati

Lo step DATA di tariffazione non scrive mai per esteso un'unica formula di attualizzazione o di credibilità inline — richiama semplicemente `pure_premium`, `blend_premium`, `charged_premium` e `pv_reserve`. Questo è il vantaggio di **PROC FCMP**: le assunzioni attuariali vivono in un'unica libreria compilata che può essere testata unitariamente, versionata e riutilizzata in lavori di tariffazione, riservazione e reporting. Cambiare la costante di credibilità `k`, il caricamento di rischio o il rapporto spese è una singola modifica nella libreria, non una caccia in ogni programma.

Leggendo il libro tariffato e l'aggregazione regionale:

- **La credibilità (`z`)** cresce con `years_insured`, esattamente come impone la formula a fluttuazione limitata `Z = sqrt(n / (n + k))`. Tra le prime dieci polizze, la polizza suburbana di un anno (10006) ottiene `z = 0.447`, la polizza urbana di due anni (10003) `z = 0.577`, mentre la polizza suburbana di nove anni (10004) raggiunge `z = 0.832`. Le polizze con poca esperienza vengono attratte verso la tariffa di classe regionale; quelle con maggiore anzianità si affidano di più ai propri sinistri.
- **La combinazione in azione:** le polizze senza sinistri (la maggior parte del libro) hanno `own_pp = 0`, quindi `blend_premium` restituisce un `blended_pp` vicino a `(1 - z)` volte la tariffa di classe — ad es. la polizza 10001 (regione rurale, `z = 0.745`) si attesta a `91.67` contro una tariffa di classe di `360`/anno-auto. Le due polizze urbane che hanno effettivamente subito sinistri, la 10002 e la 10003, portano invece il loro costo sinistri combinato verso l'alto, avvicinandolo alla propria esperienza elevata (`3039.59` e `3046.88`).
- **Il premio caricato** si colloca sopra il costo sinistri combinato perché `charged_premium` aggiunge un caricamento di rischio del 12% e maggiora per un rapporto spese del 25%, un moltiplicatore fisso `1.12 / 0.75 = 1.493` sul costo sinistri. Il premio caricato medio è `476.61` (regione rurale), `813.04` (regione suburbana) e `1,987.17` (regione urbana).
- **Riserve attualizzate:** `pv_reserve` attualizza la riserva sinistri ancora aperta di ciascuna polizza (35% dei sinistri incorsi) di tre anni al 4%, un fattore di attualizzazione di `0.889`. Le polizze senza sinistri portano `reserve_pv = 0`; i due sinistrati urbani contribuiscono con `1,067.95` e `2,226.55`. Aggregato, il libro detiene una riserva attualizzata di `$2,151.56` (regione rurale), `$5,932.52` (regione suburbana) e `$13,364.13` (regione urbana).
- **I rapporti sinistri impliciti** si attestano al 60.5% (regione rurale), 55.8% (regione suburbana) e 63.6% (regione urbana) — tutti comodamente sotto il 100%, quindi il premio caricato copre la perdita attesa in ogni segmento. Il segmento urbano è il più caldo, trainato dai suoi due grandi sinistri simulati; una revisione tariffaria reale verificherebbe se questo segnale persiste su più anni di sinistro prima di modificare la tariffa manuale.

La subroutine `blend_premium` dimostra anche l'idioma `OUTARGS` per restituire più risultati da un'unica `CALL` — il costo sinistri combinato e il fattore di credibilità `z` tornano insieme — evitando chiamate a funzioni separate e mantenendo la logica di tariffazione per polizza compatta e verificabile.